 ## Lab - Week 10 - Dropout

### Ways to reduce overfitting in neural networks
 - Getting more training data
 - Reducing the capacity of the network (last week's lab)
 - Adding weight regularization (last week's lab)
 - Adding dropout (the subject for this week)
 - Using transfer learning

### Typical training and validation loss over time
![](https://miro.medium.com/v2/resize:fit:720/format:webp/1*0VWDpLIRcMTssDf-zyOR4w.jpeg)

## Dropout Regularization

**Dropout** is a regularization technique that involves randomly "dropping out" (setting to zero) a subset of a layer's output features during training.

For example, if a layer would normally return the vector `[0.2, 0.5, 1.3, 0.8, 1.1]` for a specific input, applying dropout might result in `[0, 0.5, 1.3, 0, 1.1]`. The **dropout rate** defines the fraction of features zeroed out and typically ranges between 0.2 and 0.5.

At test time, no units are dropped. To compensate for the fact that all neurons are now active (unlike during training), the layer’s output values are scaled down by a factor equal to the dropout rate. This ensures that the expected sum of the inputs to the next layer remains consistent between training and inference.


![dropout](https://cdn-images-1.medium.com/max/1600/1*iWQzxhVlvadk6VAJjsgXgg.png)

In Keras you can introduce dropout in a network via the `Dropout` layer, which gets applied to the output of layer right before it, e.g.:
```python
model.add(layers.Dropout(0.5))
```

### Part 1: Load and Prepare Data - IMDB dataset

In [ ]:
from keras import models
from keras import layers
from keras.datasets import imdb
import numpy as np

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

def vectorize_sequences(sequences, dimension=10000):
    # Create an all-zero matrix of shape (len(sequences), dimension)
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
      for j in sequence:
        results[i, j] += 1
    return results

# Our vectorized training data
x_train = vectorize_sequences(train_data)
# Our vectorized test data
x_test = vectorize_sequences(test_data)
# Our vectorized labels
y_train = np.asarray(train_labels).astype('float32')
y_test = np.asarray(test_labels).astype('float32')

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


**As a routine, perform a sanity check on the dataset to understand the shape of the train and test inputs and outputs, and print out a sample of the data.**

In [ ]:
print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("y_train dtype:", y_train.dtype)
print("\nFirst training row (first 20 features):", x_train[0, :20])
print("First label:", y_train[0])

### Part 2: Build a Simple Neural Network Model
Objective: Create a basic neural network model using Keras.

**Questions**
- What type of classification problem are we solving?
- How many neurons should there be in the output layer?
- What should be the activation function of the output layer?
- What loss function should be used?
- What activation function should be used on the hidden layers?


Build a sequential model with one dense layer with 8 units and train it for 20 epochs.

Complete the code below:

In [ ]:
# Binary classification (positive / negative movie review).
# Output layer: 1 neuron, sigmoid activation.
# Loss: binary_crossentropy. Hidden layers: relu (standard for dense nets).

model = models.Sequential()
model.add(layers.InputLayer(shape=(10000,)))
model.add(layers.Dense(8, activation="relu"))
model.add(layers.Dense(1, activation="sigmoid"))
model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


In [ ]:
# Train the model
hist_a = model.fit(x_train, y_train, epochs=20, batch_size=512, validation_data=(x_test, y_test))

# Evaluate the model
evaluation_results = model.evaluate(x_test, y_test)
print("Test accuracy:", evaluation_results[1])

### Part 3: Visualize Training and Validation Performance

The function below can be used later to plot the loss and accuracy from the model training history

In [ ]:
# Helper function

import matplotlib.pyplot as plt
# colors will be used to plot the different models below
colors = ['blue', 'red', 'green', 'purple', 'orange', 'brown', 'pink', 'gray', 'olive', 'cyan']

def plot_history(history, color='blue', prefix=""):
    history_dict = history.history
    loss_values = history_dict["loss"]
    val_loss_values = history_dict["val_loss"]
    epochs = range(1, len(loss_values) + 1)

    # Make a figure with two subplots side by side
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)

    # Plot the loss
    plt.plot(epochs, loss_values, "o",  color=color, label=prefix + " Training loss")
    plt.plot(epochs, val_loss_values, color=color, label=prefix + " Validation loss")
    plt.title("Training and validation loss")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend(framealpha=0.5)
    plt.grid()

    # Plot the accuracy
    plt.subplot(1, 2, 2)
    acc_key = "accuracy" if "accuracy" in history_dict else "acc"
    val_acc_key = "val_accuracy" if "val_accuracy" in history_dict else "val_acc"
    acc_values = history_dict[acc_key]
    val_acc_values = history_dict[val_acc_key]
    # Skip plotting the training accuracy, it makes the plot harder to read
    # plt.plot(epochs, acc_values, "o", color=color , label=prefix + " Training accuracy")
    plt.plot(epochs, val_acc_values, color=color, label=prefix + " Validation accuracy")
    plt.title("Training and validation accuracy")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend(framealpha=0.5)
    plt.grid()

In [ ]:
# Using the plotting function

plot_history(hist_a, color='blue', prefix="Small model")

### Part 4: Add Dropout Regularization
Objective: Experiment with dropout layers in the model.

**Instructions:**
1. Add another hidden layer with 8 units
1. Add dropout layers after each hidden layer with a dropout rate of 0.3.
1. Compile the model.
1. Train the model and observe the changes in accuracy.

Complete the code below:

In [ ]:
drp_model = models.Sequential()
drp_model.add(layers.InputLayer(shape=(10000,)))
drp_model.add(layers.Dense(8, activation="relu"))
drp_model.add(layers.Dropout(0.3))
drp_model.add(layers.Dense(8, activation="relu"))
drp_model.add(layers.Dropout(0.3))
drp_model.add(layers.Dense(1, activation="sigmoid"))
drp_model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])

drp_history = drp_model.fit(
    x_train, y_train, epochs=20, batch_size=512, validation_data=(x_test, y_test)
)

In [ ]:
# Plot training history
plot_history(drp_history, color='red', prefix="Dropout 0.3")

### Part 5: Test Different Dropout Rates
Objective: Compare the model's performance with varying dropout rates.

**Instructions:**
1. Write a loop to create and train models with dropout rates of `[0, 0.2, 0.3, 0.4, 0.5]`.
2. For each model, record the history to plot it later


Complete the code below:

In [ ]:
rates = [0, 0.2, 0.3, 0.4, 0.5]
hists = []

for rate in rates:
    print(f"Training model with dropout rate: {rate}")
    model = models.Sequential()
    model.add(layers.InputLayer(shape=(10000,)))
    model.add(layers.Dense(8, activation="relu"))
    if rate > 0:
        model.add(layers.Dropout(rate))
    model.add(layers.Dense(8, activation="relu"))
    if rate > 0:
        model.add(layers.Dropout(rate))
    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])

    hist = model.fit(
        x_train, y_train, epochs=20, batch_size=512, validation_data=(x_test, y_test)
    )
    hists.append(hist)



In [ ]:
for i, hist in enumerate(hists):
    plot_history(hist, colors[i], prefix=str(rates[i]))

### Part 6: Experiment with dropout location
Objective: Compare the model's performance with dropout placed after different layers

**Instructions:**
Create 3 models with a single dropout "layer" placed:
1. Before the first dense layer only
2. After the first dense layer only
3. After the second dense layer only



Complete the code below:

In [ ]:
hists_b = []
rate = 0.4

# 1) Dropout before the first dense layer only
m1 = models.Sequential()
m1.add(layers.InputLayer(shape=(10000,)))
m1.add(layers.Dropout(rate))
m1.add(layers.Dense(8, activation="relu"))
m1.add(layers.Dense(8, activation="relu"))
m1.add(layers.Dense(1, activation="sigmoid"))
m1.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
hists_b.append(
    m1.fit(x_train, y_train, epochs=20, batch_size=512, validation_data=(x_test, y_test))
)

# 2) Dropout after the first dense layer only
m2 = models.Sequential()
m2.add(layers.InputLayer(shape=(10000,)))
m2.add(layers.Dense(8, activation="relu"))
m2.add(layers.Dropout(rate))
m2.add(layers.Dense(8, activation="relu"))
m2.add(layers.Dense(1, activation="sigmoid"))
m2.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
hists_b.append(
    m2.fit(x_train, y_train, epochs=20, batch_size=512, validation_data=(x_test, y_test))
)

# 3) Dropout after the second dense layer only
m3 = models.Sequential()
m3.add(layers.InputLayer(shape=(10000,)))
m3.add(layers.Dense(8, activation="relu"))
m3.add(layers.Dense(8, activation="relu"))
m3.add(layers.Dropout(rate))
m3.add(layers.Dense(1, activation="sigmoid"))
m3.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
hists_b.append(
    m3.fit(x_train, y_train, epochs=20, batch_size=512, validation_data=(x_test, y_test))
)


In [ ]:
prefixes = ['before 1st layer', 'after 1st layer', 'after 2nd layer']
for i, hist in enumerate(hists_b):
    plot_history(hist, colors[i], prefix=prefixes[i])